# 🌱 SAGE Inference Server (Kaggle / Colab GPU)

Serves **Qwen2.5-VL-3B-Instruct + per-crop LoRA adapters** over a public Cloudflare tunnel,
exposing the API the SAGE middleware expects:

| Method | Path | Purpose |
|---|---|---|
| GET | `/health` | liveness + active crop |
| GET | `/api/v1/crops` | crops, default prompts, known classes |
| GET | `/api/v1/model/status` | adapters + GPU memory |
| POST | `/api/v1/predict` | `crop`, `image`, optional `prompt`, `max_new_tokens`, `temperature` |

Adapters are downloaded **on demand** the first time the middleware asks for a crop, then kept in GPU memory,
so switching crops is instant after the first request.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**,
and add a secret named **`HF_TOKEN`** (Add-ons → Secrets). Optional secret **`SAGE_API_KEY`** to protect the API.
Then **Run All** and copy the `BACKEND_URL=` line printed at the end into `middleware/.env`.

In [ ]:
# 1. Install dependencies (≈1–2 min)
!pip uninstall -y -q torchao
!pip install -q -U "transformers>=4.49" "peft>=0.13" accelerate huggingface_hub fastapi "uvicorn[standard]" python-multipart pillow

In [ ]:
# 2. Configuration
import os, re, glob

# Adapter paths inside the HF repo (from modelpath.txt). Edit here if a checkpoint moves.
MODELPATH_TXT = """
Base - IDK-Chaitanya/SAGE5Class-cp
corn - corn_5ep/best_adapter
rice - rice/adapter
soybean - soybean_5ep/best_adapter
tomato - tomato/adapter
wheat - wheat/adapter
"""

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# Kaggle "Models" mount of the base weights (faster than downloading). Falls back to the HF Hub if missing.
KAGGLE_BASE_PATH = "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2"
ADAPTER_CACHE_DIR = "/kaggle/working/sage_adapters" if os.path.isdir("/kaggle/working") else "./sage_adapters"
PORT = 8000
MAX_IMAGE_MB = 10
MAX_IMAGE_SIDE = 1024        # larger uploads are downscaled to keep T4 memory safe
PRELOAD_CROPS = []           # e.g. ["tomato"] to download + attach adapters at startup


def parse_modelpath(text):
    repo, paths = None, {}
    for line in text.strip().splitlines():
        if " - " not in line:
            continue
        key, value = (s.strip() for s in line.split(" - ", 1))
        if key.lower() == "base":
            repo = value
        else:
            paths[key.lower()] = value.strip("/")
    return repo, paths


# Prefer a modelpath.txt uploaded next to the notebook / as a dataset, else the inline copy above.
_mp_files = glob.glob("modelpath.txt") + glob.glob("/kaggle/input/**/modelpath.txt", recursive=True)
if _mp_files:
    MODELPATH_TXT = open(_mp_files[0]).read()
    print("Using", _mp_files[0])
ADAPTER_REPO, ADAPTER_PATHS = parse_modelpath(MODELPATH_TXT)


def load_secret(name):
    """Kaggle Secrets → Colab userdata → environment → any .env file nearby."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    for f in glob.glob(".env") + glob.glob("/kaggle/input/**/.env", recursive=True):
        for line in open(f):
            k, _, v = line.strip().partition("=")
            if k.strip() == name and v.strip():
                return v.strip().strip('"\'')
    return None


HF_TOKEN = load_secret("HF_TOKEN")
SAGE_API_KEY = load_secret("SAGE_API_KEY") or ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Adapter repo :", ADAPTER_REPO)
print("Adapters     :", ADAPTER_PATHS)
print("HF_TOKEN     :", "found" if HF_TOKEN else "MISSING (public/slow downloads only)")
print("API key auth :", "enabled" if SAGE_API_KEY else "disabled")

In [ ]:
# 3. Crop registry: display names + classes the adapters were trained on (used for the prompt and parsing)
CROP_CLASSES = {
    "corn": ["Bacterial Rot And Blight", "Common Rust", "Ear Rots", "Gray Leaf Spot", "Leaf Blight",
             "Maize Lethal Necrosis", "Maize Streak Virus", "Northern Leaf Blight", "Healthy"],
    "rice": ["Bacterial Leaf Blight", "Blast", "Brown Spot", "Tungro", "Healthy"],
    "soybean": ["Bacterial Blight", "Bacterial Pustule", "Brown Spot", "Frogeye Leaf Spot", "Fusarium Disease",
                "Herbicide Injury", "Iron Deficiency Chlorosis", "Potassium Deficiency", "Healthy"],
    "tomato": ["Bacterial Leaf Spot", "Early Blight", "Late Blight", "Leaf Mold", "Leaf Mosaic Virus",
               "Septoria Leaf Blotch", "Target Spot", "Tomato Leaf Curl Virus", "Healthy"],
    "wheat": ["Root Rot", "Rust", "Septoria Leaf Blotch", "Smut", "Healthy"],
}
DISPLAY = {"corn": "Corn (Maize)", "rice": "Rice", "soybean": "Soybean", "tomato": "Tomato", "wheat": "Wheat"}


def default_prompt(crop):
    # Same instruction style the adapters were evaluated with.
    return (f"Identify the disease in this {crop} leaf image.\n\n"
            f"Choose exactly one class from the following list:\n{', '.join(CROP_CLASSES[crop])}.\n\n"
            "Return only the exact disease class name.\nDo not repeat the instruction.\n"
            "Do not explain the answer.\nDo not return JSON.")


CROPS = [c for c in CROP_CLASSES if c in ADAPTER_PATHS]
missing = set(ADAPTER_PATHS) - set(CROP_CLASSES)
assert not missing, f"Add class lists for: {missing}"
print("Serving crops:", CROPS)

In [ ]:
# 4. Load processor + base model
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings!"
print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

BASE_SOURCE = KAGGLE_BASE_PATH if os.path.isdir(KAGGLE_BASE_PATH) else BASE_MODEL_ID
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_SOURCE, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa", token=HF_TOKEN,
)
base_model.eval()
print("Base model loaded from", BASE_SOURCE,
      "| GPU mem:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
# 5. On-demand adapter manager
import threading, time
from huggingface_hub import snapshot_download
from peft import PeftModel

GPU_LOCK = threading.RLock()   # one generation / adapter switch at a time


class AdapterManager:
    def __init__(self, base):
        self.base = base
        self.model = None            # PeftModel once the first adapter is attached
        self.active = None
        self.loaded = []
        self.status = {c: "not_loaded" for c in CROPS}

    def local_path(self, crop):
        sub = ADAPTER_PATHS[crop]
        snapshot_download(repo_id=ADAPTER_REPO, allow_patterns=[f"{sub}/*"], local_dir=ADAPTER_CACHE_DIR, token=HF_TOKEN)
        path = os.path.join(ADAPTER_CACHE_DIR, sub)
        assert os.path.isfile(os.path.join(path, "adapter_config.json")), f"adapter_config.json missing in {path}"
        return path

    def activate(self, crop):
        """Download (first time), attach, and switch to the crop's adapter. Caller holds GPU_LOCK."""
        if crop not in ADAPTER_PATHS:
            raise ValueError(f"Unsupported crop '{crop}'. Supported: {', '.join(CROPS)}")
        if crop == self.active:
            return
        if crop not in self.loaded:
            self.status[crop] = "loading"
            t = time.time()
            try:
                path = self.local_path(crop)
                if self.model is None:
                    self.model = PeftModel.from_pretrained(self.base, path, adapter_name=crop, is_trainable=False)
                else:
                    self.model.load_adapter(path, adapter_name=crop, is_trainable=False)
            except Exception:
                self.status[crop] = "error"
                raise
            self.loaded.append(crop)
            self.status[crop] = "loaded"
            print(f"[adapter] {crop} attached in {time.time() - t:.1f}s")
        self.model.set_adapter(crop)
        self.model.eval()
        self.active = crop
        print(f"[adapter] active = {crop}")

    def runtime(self):
        return self.model if self.model is not None else self.base


adapters = AdapterManager(base_model)
for c in PRELOAD_CROPS:
    with GPU_LOCK:
        adapters.activate(c)

In [ ]:
# 6. Inference + response parsing
import io
from PIL import Image, ImageOps


def norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


def match_class(text, crop):
    """Exact class match first, then the first class name mentioned anywhere in the output."""
    classes = CROP_CLASSES[crop]
    cleaned = text.strip().strip("\"'` .")
    m = re.search(r"(?im)^\s*(?:predicted\s+disease|disease|answer|class|class_name)\s*[:=]\s*(.+?)\s*$", text)
    for cand in filter(None, [cleaned, m and m.group(1).strip("\"'` .")]):
        for cls in classes:
            if norm(cand) == norm(cls):
                return cls, True
    hits = [(norm(text).find(norm(cls)), cls) for cls in classes if norm(cls) in norm(text)]
    if hits:
        return min(hits)[1], True
    return None, False


def load_image(data):
    with Image.open(io.BytesIO(data)) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
    if max(im.size) > MAX_IMAGE_SIDE:
        im.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    return im


def predict(crop, image_bytes, prompt=None, max_new_tokens=256, temperature=0.0):
    crop = crop.strip().lower()
    prompt = (prompt or "").strip() or default_prompt(crop)
    with Image.open(io.BytesIO(image_bytes)) as probe:
        fmt, (w, h) = probe.format, probe.size
    image = load_image(image_bytes)

    with GPU_LOCK:
        adapters.activate(crop)
        model = adapters.runtime()
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt")
        inputs = inputs.to(model.get_input_embeddings().weight.device)
        gen = {"max_new_tokens": int(max_new_tokens), "use_cache": True}
        if temperature and temperature > 0:
            gen.update(do_sample=True, temperature=float(temperature))
        else:
            gen.update(do_sample=False)
        t = time.time()
        with torch.inference_mode():
            out = model.generate(**inputs, **gen)
        elapsed = time.time() - t
        n_in = inputs["input_ids"].shape[1]
        text = processor.batch_decode(out[:, n_in:], skip_special_tokens=True,
                                      clean_up_tokenization_spaces=False)[0].strip()

    cls, canonical = match_class(text, crop)
    return {
        "success": True,
        "crop": crop,
        "base_model": BASE_MODEL_ID,
        "active_adapter": crop,
        "checkpoint": f"{ADAPTER_REPO}/{ADAPTER_PATHS[crop]}",
        "input": {"prompt": prompt},
        "prediction": {
            "response": text,
            "class_name": cls or (text.splitlines()[0][:80] if text else None),
            "crop": crop,
            "disease_type": None, "plant_organ": "Leaf", "visual_symptoms": None, "pathogen": None,
            "matched_taxonomy_class": cls,
            "is_canonical_match": canonical,
            "attributes": {"generated_tokens": int(out.shape[1] - n_in)},
        },
        "raw_output": text,
        "inference_time_seconds": round(elapsed, 3),
        "image_metadata": {"width": w, "height": h, "format": fmt},
    }

In [ ]:
# 7. FastAPI app (same contract the middleware already uses)
from typing import Optional
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile

ALLOWED = {"image/jpeg", "image/jpg", "image/png", "image/webp"}
app = FastAPI(title="SAGE Inference Server")


def check_key(authorization: Optional[str], x_api_key: Optional[str]):
    if not SAGE_API_KEY:
        return
    bearer = (authorization or "").removeprefix("Bearer ").strip()
    if SAGE_API_KEY not in (bearer, (x_api_key or "").strip()):
        raise HTTPException(401, "Invalid or missing API key.")


def gpu_status():
    if not torch.cuda.is_available():
        return {"available": False}
    gb = 1024 ** 3
    return {"available": True, "device_name": torch.cuda.get_device_name(0),
            "total_memory_gb": round(sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / gb, 2),
            "allocated_memory_gb": round(sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / gb, 2),
            "reserved_memory_gb": round(sum(torch.cuda.memory_reserved(i) for i in range(torch.cuda.device_count())) / gb, 2)}


@app.get("/health")
def health():
    return {"status": "healthy", "service": "SAGE API", "device": "cuda", "base_model_loaded": True,
            "active_crop": adapters.active}


@app.get("/api/v1/crops")
def crops(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    details = [{
        "crop_id": c, "display_name": DISPLAY.get(c, c.title()), "hf_repo": ADAPTER_REPO,
        "checkpoint_subpath": ADAPTER_PATHS[c], "model_type": "peft_lora", "adapter_name": c,
        "default_prompt": default_prompt(c), "loading_status": adapters.status[c],
        "classes_count": len(CROP_CLASSES[c]), "known_classes": CROP_CLASSES[c],
    } for c in CROPS]
    return {"total_crops": len(CROPS), "available_crops": CROPS, "crop_details": details}


@app.get("/api/v1/model/status")
def model_status(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    return {"base_model": BASE_MODEL_ID, "active_crop": adapters.active, "active_adapter": adapters.active,
            "adapter_mode": "peft_multi_adapter", "base_model_loaded": True, "loaded_adapters": adapters.loaded,
            "gpu_status": gpu_status(), "mock_mode": False}


@app.post("/api/v1/predict")
def predict_route(crop: str = Form(...), image: UploadFile = File(...), prompt: Optional[str] = Form(None),
                  max_new_tokens: Optional[int] = Form(None), temperature: Optional[float] = Form(None),
                  authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    if crop.strip().lower() not in ADAPTER_PATHS:
        raise HTTPException(400, f"Unsupported crop '{crop}'. Supported: {', '.join(CROPS)}")
    if (image.content_type or "").lower() not in ALLOWED:
        raise HTTPException(415, f"Unsupported image type '{image.content_type}'. Use JPEG, PNG or WebP.")
    data = image.file.read()
    if not data:
        raise HTTPException(400, "Uploaded image is empty.")
    if len(data) > MAX_IMAGE_MB * 1024 * 1024:
        raise HTTPException(413, f"Image exceeds {MAX_IMAGE_MB} MB.")
    try:
        Image.open(io.BytesIO(data)).verify()
    except Exception:
        raise HTTPException(422, "Uploaded file is corrupted or not a valid image.")
    try:
        return predict(crop, data, prompt, max_new_tokens or 256, temperature or 0.0)
    except ValueError as e:
        raise HTTPException(400, str(e))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(500, "GPU out of memory — try a smaller image or fewer max_new_tokens.")
    except Exception as e:
        raise HTTPException(500, f"Inference failed: {e}")

In [ ]:
# 8. Start the server + public Cloudflare tunnel
import subprocess, uvicorn, urllib.request, json as _json

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
def http_get(url, headers=None, timeout=15):
    req = urllib.request.Request(url, headers={"User-Agent": "sage-notebook", **(headers or {})})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, _json.loads(r.read())

for _ in range(60):
    try:
        http_get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        break
    except Exception as e:
        last_err = e
        time.sleep(1)
else:
    raise RuntimeError(f"Local server did not start: {last_err!r}")
print("Local server is up on port", PORT)

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining logs

print("\n" + "=" * 70)
print("  SAGE backend is live:", PUBLIC_URL)
print("  Paste into middleware/.env and restart the middleware:\n")
print(f"  BACKEND_URL={PUBLIC_URL}")
if SAGE_API_KEY:
    print("  BACKEND_API_KEY=<the SAGE_API_KEY secret you set>")
print("=" * 70)

In [ ]:
# 9. Smoke test through the public URL (tunnel DNS can take ~10–30 s to propagate)
hdr = {"Authorization": f"Bearer {SAGE_API_KEY}"} if SAGE_API_KEY else {}
for attempt in range(10):
    try:
        status, body = http_get(f"{PUBLIC_URL}/api/v1/model/status", headers=hdr)
        print("Tunnel OK:", status, body)
        break
    except Exception as e:
        print("waiting for tunnel…", repr(e)[:200])
        time.sleep(5)

In [ ]:
# 10. Keep the session alive and log status every 5 min (interrupt this cell to stop logging; the server keeps running)
try:
    while True:
        g = gpu_status()
        print(time.strftime("%H:%M:%S"), "| active:", adapters.active, "| loaded:", adapters.loaded,
              "| GPU alloc:", g.get("allocated_memory_gb"), "GB |", PUBLIC_URL)
        time.sleep(300)
except KeyboardInterrupt:
    print("Logging stopped; server still running.")